## 공개용 코드 안내

이 노트북은 실제 Colab 실험 노트북에서 셀 출력(이미지·실행 로그)을 제거한 코드입니다. 데이터 ZIP 비밀번호는 `VQA_ZIP_PASSWORD` 환경변수에서 읽도록 변경했습니다. 데이터/LoRA 가중치는 포함하지 않습니다. 경로는 본인의 Colab Drive 구성에 맞게 수정해야 합니다.



# Qwen3-VL-8B VQA — V3: 1536 + Question-Aware OCR Crop

> **목표:** 현재 최고 성능인 `1536 visual tokens / OCR text 없음`을 기반으로,  
> OCR은 정답 힌트를 텍스트로 넣는 대신 **질문과 관련된 글자 영역을 찾는 locator로만 사용**합니다.

## 지금까지의 실험

| Version | Visual Tokens | OCR | Kaggle Public |
|---|---:|---|---:|
| V1 | 1024 | X | 0.95770 |
| **V1.1** | **1536** | **X** | **0.95859** |
| V2-A | 2048 | X | 0.95770 |
| V2-B | 2048 | OCR Text | 0.95442 |

### 해석

- `1024 → 1536`: 소폭 상승
- `1536 → 2048`: 상승하지 않음
- OCR text 전체 삽입: 성능 하락

따라서 V3는:

```text
Visual Tokens = 1536
OCR Text Prompt = 사용 안 함
OCR Bounding Box = 관련 영역 탐색에만 사용
```

합니다.

---

## V3 파이프라인

```text
Full Image
    │
    ├──────────────────────────► Qwen3-VL
    │
    ▼
Local OCR
    │
    ▼
text + bounding boxes
    │
Question + Choices와 관련도 계산
    │
    ▼
Best OCR region
    │
    ▼
주변까지 넓게 Crop
    │
    └──────────────────────────► Qwen3-VL
                                     │
                            Full Image + Crop
                                     │
                            Question + Choices
                                     │
                                     ▼
                                  a/b/c/d
```

OCR text가 틀리더라도 텍스트 자체를 모델에게 주지 않고 **위치 탐색에만 활용**합니다.

---

## 처음 실행

```python
DEBUG = True
```

로 전체 파이프라인을 먼저 확인합니다. 정상 동작 후:

```python
DEBUG = False
```

로 변경하고 런타임을 재시작한 뒤 전체 학습하세요.


# 0. GPU 확인

In [ ]:
!nvidia-smi


# 1. 라이브러리 설치

설치 후 필요하면 `런타임 → 세션 다시 시작`을 한 번 해주세요.

`torchao 0.10.0`과 PEFT 충돌을 피하기 위해 V3에서 사용하지 않는 torchao는 제거합니다.


In [ ]:
%pip install -q -U \
    "transformers>=4.57.0,<5" \
    "accelerate>=1.0.0" \
    "peft>=0.13.2" \
    "huggingface_hub>=0.25.0,<1.0" \
    "qwen_vl_utils>=0.0.14"

%pip install -q \
    "pandas==2.2.2" \
    "pillow==11.3.0" \
    "easyocr==1.7.2" \
    tqdm psutil matplotlib

%pip uninstall -y torchao


# 2. 기본 설정

In [ ]:
import os
import gc
import re
import math
import json
import random
import shutil
from collections import Counter
from dataclasses import dataclass
from difflib import SequenceMatcher
from typing import Any, Dict, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
SEED = 42

# 최초 테스트는 True, 최종 전체 학습은 False
DEBUG = False

DEBUG_TRAIN_SAMPLES = 100
DEBUG_DEV_SAMPLES = 50
DEBUG_TEST_SAMPLES = 50

# 현재 최고점 설정
MIN_VISUAL_TOKENS = 256
MAX_VISUAL_TOKENS = 1536

MIN_PIXELS = MIN_VISUAL_TOKENS * 32 * 32
MAX_PIXELS = MAX_VISUAL_TOKENS * 32 * 32

# -----------------------------
# Question-aware OCR Crop
# -----------------------------
USE_OCR_CROP = True

OCR_LANGS = ["ko", "en"]
OCR_MIN_CONF = 0.20

# OCR box 주변까지 넓게 포함
CROP_EXPAND_X = 2.3
CROP_EXPAND_Y = 3.0

MIN_CROP_WIDTH = 480
MIN_CROP_HEIGHT = 320

# 관련도가 너무 낮으면 crop을 사용하지 않음
MIN_RELEVANCE_SCORE = 0.18

# -----------------------------
# Training
# -----------------------------
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 4

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

NUM_WORKERS = 0

EXPECTED_FULL_TRAIN_ROWS = 6714
EXPECTED_FULL_DEV_ROWS = 2683
EXPECTED_FULL_TEST_ROWS = 6714

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
Image.MAX_IMAGE_PIXELS = None

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다."
assert torch.cuda.is_bf16_supported(), "BF16 GPU가 필요합니다."

print("Torch:", torch.__version__)
print("CUDA :", torch.version.cuda)
print("GPU  :", torch.cuda.get_device_name(0))
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory/(1024**3), 1), "GB")
print()
print("DEBUG             :", DEBUG)
print("USE_OCR_CROP      :", USE_OCR_CROP)
print("MAX_VISUAL_TOKENS :", MAX_VISUAL_TOKENS)
print("MAX_PIXELS        :", MAX_PIXELS, f"(~{MAX_PIXELS/1_000_000:.2f} MP)")


# 3. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")



# 4. 데이터 / 결과 경로

V2에서 이미 OCR cache를 생성했다면 그대로 재사용합니다.

```text
MyDrive/Qwen3VL_VQA_V2_2048_OCR/ocr_cache/
```

에 `ocr_train.csv`, `ocr_dev.csv`, `ocr_test.csv`가 있으면 새 OCR을 돌리지 않습니다.


In [ ]:
EXTRACT_ZIP = True

ZIP_PATH = "/content/drive/MyDrive/AI Chal/ssafy-16-2-ai.zip"
ZIP_PASSWORD = os.getenv("VQA_ZIP_PASSWORD", "")  # 비밀번호는 환경변수로 관리

DATA_ROOT = "/content"

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Qwen3VL_VQA_V3_1536_OCR_CROP"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

PREVIOUS_OCR_CACHE_DIR = "/content/drive/MyDrive/Qwen3VL_VQA_V2_2048_OCR/ocr_cache"

LOCAL_OCR_CACHE_DIR = os.path.join(
    DRIVE_PROJECT_DIR,
    "ocr_cache",
)
os.makedirs(LOCAL_OCR_CACHE_DIR, exist_ok=True)

if EXTRACT_ZIP:
    if not ZIP_PASSWORD:
        raise ValueError("VQA_ZIP_PASSWORD 환경변수를 설정하거나 EXTRACT_ZIP=False로 변경하세요.")
    assert os.path.isfile(ZIP_PATH), ZIP_PATH
    rc = os.system(
        f'7z x -y -p"{ZIP_PASSWORD}" -o"{DATA_ROOT}" "{ZIP_PATH}"'
    )
    assert rc == 0

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT   :", DRIVE_PROJECT_DIR)
print("V2 OCR cache:", PREVIOUS_OCR_CACHE_DIR)


# 5. CSV 로드 및 검증

In [ ]:
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
DEV_CSV = os.path.join(DATA_ROOT, "dev.csv")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")

for p in [TRAIN_CSV, DEV_CSV, TEST_CSV]:
    assert os.path.isfile(p), p

train_full_df = pd.read_csv(TRAIN_CSV)
dev_full_df = pd.read_csv(DEV_CSV)
test_full_df = pd.read_csv(TEST_CSV)

assert len(train_full_df) == EXPECTED_FULL_TRAIN_ROWS
assert len(dev_full_df) == EXPECTED_FULL_DEV_ROWS
assert len(test_full_df) == EXPECTED_FULL_TEST_ROWS

required = {"id", "path", "question", "a", "b", "c", "d"}
assert required.issubset(train_full_df.columns)
assert required.issubset(dev_full_df.columns)
assert required.issubset(test_full_df.columns)
assert "answer" in train_full_df.columns

DEV_ANSWER_COLS = [
    c for c in ["answer1", "answer2", "answer3", "answer4", "answer5"]
    if c in dev_full_df.columns
]

train_full_df["answer"] = (
    train_full_df["answer"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if DEBUG:
    train_df = train_full_df.sample(
        min(DEBUG_TRAIN_SAMPLES, len(train_full_df)),
        random_state=SEED,
    ).reset_index(drop=True)
    dev_df = dev_full_df.head(
        min(DEBUG_DEV_SAMPLES, len(dev_full_df))
    ).reset_index(drop=True)
    test_df = test_full_df.head(
        min(DEBUG_TEST_SAMPLES, len(test_full_df))
    ).reset_index(drop=True)
else:
    train_df = train_full_df.reset_index(drop=True)
    dev_df = dev_full_df.reset_index(drop=True)
    test_df = test_full_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Dev  :", len(dev_df))
print("Test :", len(test_df))


In [ ]:
def resolve_image_path(path):
    path = str(path)
    if os.path.isabs(path):
        return path
    return os.path.join(DATA_ROOT, path)

for split_name, df in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df),
]:
    missing = [
        resolve_image_path(p)
        for p in df["path"].astype(str)
        if not os.path.isfile(resolve_image_path(p))
    ]
    print(split_name, "missing:", len(missing))
    assert not missing



# 6. OCR Bounding Box Cache

V3는 OCR text를 Qwen prompt에 넣지 않습니다.

다만 OCR이 검출한:

```text
text
confidence
bbox
```

중 `bbox`를 이용해 관련 영역을 확대합니다.

V2 cache가 있으면 그대로 재사용하고, 없으면 EasyOCR을 로컬에서 실행합니다.


In [ ]:
def choose_cache_path(split):
    prev = os.path.join(
        PREVIOUS_OCR_CACHE_DIR,
        f"ocr_{split}.csv",
    )
    local = os.path.join(
        LOCAL_OCR_CACHE_DIR,
        f"ocr_{split}.csv",
    )

    if os.path.isfile(prev):
        print(f"{split}: V2 OCR cache 재사용")
        return prev
    return local

OCR_TRAIN_PATH = choose_cache_path("train")
OCR_DEV_PATH = choose_cache_path("dev")
OCR_TEST_PATH = choose_cache_path("test")

print(OCR_TRAIN_PATH)
print(OCR_DEV_PATH)
print(OCR_TEST_PATH)


In [ ]:
def load_cache(path):
    if os.path.isfile(path):
        df = pd.read_csv(path)
        df["id"] = df["id"].astype(str)
        return df

    return pd.DataFrame(
        columns=["id", "ocr_text", "ocr_count", "ocr_json"]
    )


def generate_ocr_cache(df, cache_path, split_name, reader):
    cache_df = load_cache(cache_path)
    cached_ids = set(cache_df["id"].astype(str))

    todo = df[
        ~df["id"].astype(str).isin(cached_ids)
    ].copy()

    print(
        f"{split_name}: required={len(df)}, "
        f"cached={len(cache_df)}, new={len(todo)}"
    )

    new_rows = []

    for _, row in tqdm(
        todo.iterrows(),
        total=len(todo),
        desc=f"OCR {split_name}",
    ):
        path = resolve_image_path(row["path"])

        try:
            results = reader.readtext(
                path,
                detail=1,
                paragraph=False,
                decoder="greedy",
                batch_size=8,
                workers=0,
                canvas_size=3200,
                mag_ratio=1.5,
            )

            serializable = []

            for bbox, text, conf in results:
                serializable.append({
                    "bbox": [
                        [float(p[0]), float(p[1])]
                        for p in bbox
                    ],
                    "text": str(text),
                    "confidence": float(conf),
                })

            new_rows.append({
                "id": str(row["id"]),
                "ocr_text": "",
                "ocr_count": len(results),
                "ocr_json": json.dumps(
                    serializable,
                    ensure_ascii=False,
                ),
            })

        except Exception as e:
            print("OCR error:", row["id"], repr(e))
            new_rows.append({
                "id": str(row["id"]),
                "ocr_text": "",
                "ocr_count": 0,
                "ocr_json": "[]",
            })

        if len(new_rows) >= 100:
            cache_df = pd.concat(
                [cache_df, pd.DataFrame(new_rows)],
                ignore_index=True,
            ).drop_duplicates("id", keep="last")

            cache_df.to_csv(
                cache_path,
                index=False,
                encoding="utf-8-sig",
            )
            new_rows = []

    if new_rows:
        cache_df = pd.concat(
            [cache_df, pd.DataFrame(new_rows)],
            ignore_index=True,
        ).drop_duplicates("id", keep="last")

        cache_df.to_csv(
            cache_path,
            index=False,
            encoding="utf-8-sig",
        )

    return cache_df


if USE_OCR_CROP:
    missing_cache = any(
        not os.path.isfile(p)
        for p in [
            OCR_TRAIN_PATH,
            OCR_DEV_PATH,
            OCR_TEST_PATH,
        ]
    )

    if missing_cache:
        import easyocr

        reader = easyocr.Reader(
            OCR_LANGS,
            gpu=True,
            verbose=True,
        )

        if not os.path.isfile(OCR_TRAIN_PATH):
            OCR_TRAIN_PATH = os.path.join(
                LOCAL_OCR_CACHE_DIR,
                "ocr_train.csv",
            )
            generate_ocr_cache(
                train_df,
                OCR_TRAIN_PATH,
                "train",
                reader,
            )

        if not os.path.isfile(OCR_DEV_PATH):
            OCR_DEV_PATH = os.path.join(
                LOCAL_OCR_CACHE_DIR,
                "ocr_dev.csv",
            )
            generate_ocr_cache(
                dev_df,
                OCR_DEV_PATH,
                "dev",
                reader,
            )

        if not os.path.isfile(OCR_TEST_PATH):
            OCR_TEST_PATH = os.path.join(
                LOCAL_OCR_CACHE_DIR,
                "ocr_test.csv",
            )
            generate_ocr_cache(
                test_df,
                OCR_TEST_PATH,
                "test",
                reader,
            )

        del reader
        gc.collect()
        torch.cuda.empty_cache()

train_ocr_df = load_cache(OCR_TRAIN_PATH)
dev_ocr_df = load_cache(OCR_DEV_PATH)
test_ocr_df = load_cache(OCR_TEST_PATH)

print(
    "OCR cache rows:",
    len(train_ocr_df),
    len(dev_ocr_df),
    len(test_ocr_df),
)



# 7. Question-aware OCR Region Scoring

OCR box 후보의 관련도를 다음 신호로 계산합니다.

- OCR text ↔ 질문 유사도
- OCR text ↔ 각 선택지 유사도
- 숫자/전화번호/가격 ↔ 선택지 숫자 유사도
- OCR confidence

**선택 과정에서만 OCR 문자열을 사용하고 Qwen prompt에는 전달하지 않습니다.**


In [ ]:
def normalize_for_match(text):
    text = str(text).lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[^\w가-힣0-9]", "", text)
    return text


def extract_digits(text):
    return re.sub(r"\D", "", str(text))


def seq_sim(a, b):
    a = normalize_for_match(a)
    b = normalize_for_match(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def ocr_relevance_score(ocr_text, confidence, row):
    question = str(row["question"])
    choices = [
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"]),
    ]

    q_score = seq_sim(
        ocr_text,
        question,
    )

    choice_score = max(
        seq_sim(ocr_text, choice)
        for choice in choices
    )

    ocr_digits = extract_digits(ocr_text)
    digit_score = 0.0

    if len(ocr_digits) >= 2:
        for choice in choices:
            choice_digits = extract_digits(choice)

            if not choice_digits:
                continue

            digit_score = max(
                digit_score,
                SequenceMatcher(
                    None,
                    ocr_digits,
                    choice_digits,
                ).ratio(),
            )

            if ocr_digits == choice_digits:
                digit_score = 1.0

    base = (
        0.50 * choice_score
        + 0.30 * q_score
        + 0.20 * digit_score
    )

    conf_factor = (
        0.75
        + 0.25 * max(
            0.0,
            min(float(confidence), 1.0),
        )
    )

    return base * conf_factor


def parse_ocr_json(value):
    if pd.isna(value):
        return []

    try:
        return json.loads(str(value))
    except Exception:
        return []


def build_json_map(df):
    result = {}

    for _, row in df.iterrows():
        result[str(row["id"])] = parse_ocr_json(
            row.get("ocr_json", "[]")
        )

    return result


OCR_JSON_MAP = {}
OCR_JSON_MAP.update(build_json_map(train_ocr_df))
OCR_JSON_MAP.update(build_json_map(dev_ocr_df))
OCR_JSON_MAP.update(build_json_map(test_ocr_df))


def bbox_xyxy(bbox):
    xs = [float(p[0]) for p in bbox]
    ys = [float(p[1]) for p in bbox]

    return (
        min(xs),
        min(ys),
        max(xs),
        max(ys),
    )


def expanded_crop_box(bbox, image_width, image_height):
    x1, y1, x2, y2 = bbox_xyxy(bbox)

    bw = max(x2 - x1, 1)
    bh = max(y2 - y1, 1)

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    target_w = max(
        bw * CROP_EXPAND_X,
        MIN_CROP_WIDTH,
    )

    target_h = max(
        bh * CROP_EXPAND_Y,
        MIN_CROP_HEIGHT,
    )

    nx1 = max(
        0,
        int(cx - target_w / 2),
    )
    ny1 = max(
        0,
        int(cy - target_h / 2),
    )
    nx2 = min(
        image_width,
        int(cx + target_w / 2),
    )
    ny2 = min(
        image_height,
        int(cy + target_h / 2),
    )

    return (
        nx1,
        ny1,
        nx2,
        ny2,
    )


def get_question_aware_crop(row, image):
    if not USE_OCR_CROP:
        return None, {
            "used": False,
            "reason": "disabled",
        }

    items = OCR_JSON_MAP.get(
        str(row["id"]),
        [],
    )

    candidates = []

    for item in items:
        text = str(
            item.get("text", "")
        )

        conf = float(
            item.get("confidence", 0.0)
        )

        bbox = item.get("bbox")

        if (
            not text.strip()
            or conf < OCR_MIN_CONF
            or not bbox
        ):
            continue

        score = ocr_relevance_score(
            text,
            conf,
            row,
        )

        candidates.append({
            "score": score,
            "text": text,
            "confidence": conf,
            "bbox": bbox,
        })

    if not candidates:
        return None, {
            "used": False,
            "reason": "no_ocr",
        }

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True,
    )

    best = candidates[0]

    if best["score"] < MIN_RELEVANCE_SCORE:
        return None, {
            "used": False,
            "reason": "low_relevance",
            "best_score": best["score"],
            "best_text": best["text"],
        }

    crop_box = expanded_crop_box(
        best["bbox"],
        image.width,
        image.height,
    )

    crop = image.crop(
        crop_box
    )

    return crop, {
        "used": True,
        "score": best["score"],
        "ocr_text_for_selection_only": best["text"],
        "confidence": best["confidence"],
        "crop_box": crop_box,
    }



# 8. Crop 샘플 확인

**전체 학습 전에 이 셀 결과를 꼭 눈으로 확인하세요.**

선택된 영역이 질문과 무관한 경우가 많다면 전체 학습을 시작하지 말고  
`MIN_RELEVANCE_SCORE`, crop 확장 비율 등을 조정해야 합니다.


In [ ]:
import matplotlib.pyplot as plt

for i in range(min(5, len(train_df))):
    row = train_df.iloc[i]

    image = Image.open(
        resolve_image_path(row["path"])
    ).convert("RGB")

    crop, meta = get_question_aware_crop(
        row,
        image,
    )

    print("=" * 80)
    print("ID:", row["id"])
    print("Question:", row["question"])
    print(
        "Choices:",
        row["a"], "/",
        row["b"], "/",
        row["c"], "/",
        row["d"],
    )
    print("Meta:", meta)

    if crop is not None:
        plt.figure(figsize=(9, 5))
        plt.imshow(crop)
        plt.axis("off")
        plt.title("Question-aware Crop")
        plt.show()
    else:
        print("Crop 미사용")


# 9. Processor / Model / LoRA

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

processor.image_processor.size = {
    "longest_edge": MAX_PIXELS,
    "shortest_edge": MIN_PIXELS,
}

gc.collect()
torch.cuda.empty_cache()

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda")

model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(
    model,
    lora_config,
)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()

print(
    "VRAM:",
    round(
        torch.cuda.memory_allocated()/(1024**3),
        2,
    ),
    "GB",
)



# 10. Prompt / Multi-image 입력

관련 crop이 있으면:

```text
Image 1 = Full Image
Image 2 = Relevant Crop
```

을 같이 넣습니다.

OCR 문자열은 prompt에 들어가지 않습니다.


In [ ]:
SYSTEM_INSTRUCT = (
    "You are a visual multiple-choice question answering assistant. "
    "Inspect all provided images carefully. "
    "The first image is the full scene. "
    "If a second image is present, it is an enlarged region that may contain "
    "text or details relevant to the question. "
    "Pay close attention to Korean/English text, numbers, phone numbers, "
    "prices, signs, menus, labels, and tables. "
    "Choose the best answer from a, b, c, or d. "
    "Output exactly one lowercase letter and nothing else."
)


def build_mc_prompt(row):
    return (
        f"질문: {row['question']}\n\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "첫 번째 전체 이미지와, 제공된 경우 두 번째 확대 이미지를 함께 확인하세요. "
        "정답을 a, b, c, d 중 하나의 소문자 한 글자로만 답하세요."
    )


def build_messages(row, include_answer=False, answer=None):
    full_image = Image.open(
        resolve_image_path(row["path"])
    ).convert("RGB")

    crop_image, crop_meta = get_question_aware_crop(
        row,
        full_image,
    )

    images = [full_image]

    user_content = [
        {
            "type": "image",
            "image": full_image,
        }
    ]

    if crop_image is not None:
        images.append(crop_image)
        user_content.append({
            "type": "image",
            "image": crop_image,
        })

    user_content.append({
        "type": "text",
        "text": build_mc_prompt(row),
    })

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_INSTRUCT,
                }
            ],
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    if include_answer:
        gold = str(answer).strip().lower()

        messages.append({
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": gold,
                }
            ],
        })

    return images, messages, crop_meta


# 11. Dataset / Answer-only Loss

In [ ]:
class TrainVQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        gold = str(row["answer"]).strip().lower()

        images, messages, meta = build_messages(
            row,
            include_answer=True,
            answer=gold,
        )

        return {
            "images": images,
            "messages": messages,
            "gold": gold,
        }


def find_last_subsequence(sequence, subsequence, valid_len=None):
    if valid_len is None:
        valid_len = len(sequence)

    seq = list(sequence[:valid_len])
    sub = list(subsequence)

    if not sub:
        return None

    for start in range(
        valid_len - len(sub),
        -1,
        -1,
    ):
        if seq[start:start + len(sub)] == sub:
            return start

    return None


@dataclass
class AnswerOnlyMultiImageCollator:
    processor: Any

    def __call__(self, batch):
        texts = []
        flat_images = []
        golds = []

        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )

            texts.append(text)
            flat_images.extend(sample["images"])
            golds.append(sample["gold"])

        enc = self.processor(
            text=texts,
            images=flat_images,
            padding=True,
            return_tensors="pt",
        )

        labels = torch.full_like(
            enc["input_ids"],
            -100,
        )

        for i, gold in enumerate(golds):
            gold_ids = self.processor.tokenizer(
                gold,
                add_special_tokens=False,
            ).input_ids

            valid_positions = torch.nonzero(
                enc["attention_mask"][i],
                as_tuple=False,
            ).flatten()

            valid_len = int(
                valid_positions[-1].item()
            ) + 1

            pos = find_last_subsequence(
                enc["input_ids"][i].tolist(),
                gold_ids,
                valid_len,
            )

            if pos is None:
                raise RuntimeError(
                    f"정답 token을 찾지 못했습니다: {gold}"
                )

            labels[
                i,
                pos:pos + len(gold_ids)
            ] = enc["input_ids"][
                i,
                pos:pos + len(gold_ids)
            ]

        enc["labels"] = labels
        return enc


train_ds = TrainVQADataset(train_df)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=AnswerOnlyMultiImageCollator(processor),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

check_batch = next(iter(train_loader))

print(
    "Supervised tokens:",
    int(
        (check_batch["labels"] != -100)
        .sum()
        .item()
    )
)

del check_batch
gc.collect()
torch.cuda.empty_cache()


# 12. a/b/c/d Logit Scoring

In [ ]:
CHOICES = ["a", "b", "c", "d"]

choice_token_ids = {
    ch: processor.tokenizer(
        ch,
        add_special_tokens=False,
    ).input_ids
    for ch in CHOICES
}

assert all(
    len(v) == 1
    for v in choice_token_ids.values()
)


def move_to_gpu(batch):
    return {
        k: (
            v.to("cuda", non_blocking=True)
            if torch.is_tensor(v)
            else v
        )
        for k, v in batch.items()
    }


@torch.inference_mode()
def predict_one(row, model, processor):
    images, messages, crop_meta = build_messages(
        row,
        include_answer=False,
    )

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=images,
        return_tensors="pt",
    )

    inputs = move_to_gpu(inputs)

    outputs = model(**inputs)

    last_pos = int(
        torch.nonzero(
            inputs["attention_mask"][0],
            as_tuple=False,
        )[-1].item()
    )

    next_logits = outputs.logits[
        0,
        last_pos,
    ]

    scores = {
        ch: float(
            next_logits[
                choice_token_ids[ch][0]
            ].float().item()
        )
        for ch in CHOICES
    }

    pred = max(
        scores,
        key=scores.get,
    )

    return pred, scores, crop_meta


# 13. Dev 평가

In [ ]:
def get_valid_dev_labels(row):
    labels = []

    for c in DEV_ANSWER_COLS:
        value = row[c]

        if pd.notna(value):
            value = str(value).strip().lower()

            if value in CHOICES:
                labels.append(value)

    return labels


def dev_label_info(row):
    labels = get_valid_dev_labels(row)

    if not labels:
        return {
            "labels": [],
            "majority": None,
            "majority_ratio": np.nan,
            "unique_majority": False,
        }

    counts = Counter(labels)
    max_count = max(counts.values())

    winners = [
        k
        for k, v in counts.items()
        if v == max_count
    ]

    return {
        "labels": labels,
        "majority": (
            winners[0]
            if len(winners) == 1
            else None
        ),
        "majority_ratio": max_count / len(labels),
        "unique_majority": len(winners) == 1,
    }


@torch.inference_mode()
def evaluate_dev(model, processor, df, desc="Dev"):
    model.eval()
    model.config.use_cache = True

    rows = []

    consensus_scores = []
    majority_hits = []
    high_conf_hits = []

    crop_used = 0

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=desc,
    ):
        pred, scores, crop_meta = predict_one(
            row,
            model,
            processor,
        )

        if crop_meta.get("used", False):
            crop_used += 1

        info = dev_label_info(row)
        labels = info["labels"]

        consensus = (
            sum(x == pred for x in labels) / len(labels)
            if labels
            else np.nan
        )

        if labels:
            consensus_scores.append(consensus)

        majority_hit = np.nan

        if info["unique_majority"]:
            majority_hit = int(
                pred == info["majority"]
            )
            majority_hits.append(majority_hit)

        high_conf_hit = np.nan

        if (
            info["unique_majority"]
            and info["majority_ratio"] >= 0.60
        ):
            high_conf_hit = int(
                pred == info["majority"]
            )
            high_conf_hits.append(high_conf_hit)

        rows.append({
            "id": row["id"],
            "prediction": pred,
            "majority": info["majority"],
            "majority_ratio": info["majority_ratio"],
            "consensus_score": consensus,
            "majority_hit": majority_hit,
            "high_conf_hit": high_conf_hit,
            "crop_used": crop_meta.get("used", False),
            "crop_score": crop_meta.get("score", np.nan),
            # 디버깅/분석용. 모델 prompt에는 전달 안 됨.
            "crop_anchor_ocr": crop_meta.get(
                "ocr_text_for_selection_only",
                "",
            ),
            "score_a": scores["a"],
            "score_b": scores["b"],
            "score_c": scores["c"],
            "score_d": scores["d"],
        })

    metrics = {
        "consensus_score": float(np.mean(consensus_scores)),
        "majority_accuracy": float(np.mean(majority_hits)),
        "majority_n": len(majority_hits),
        "high_conf_accuracy": float(np.mean(high_conf_hits)),
        "high_conf_n": len(high_conf_hits),
        "crop_usage_rate": crop_used / len(df),
    }

    model.config.use_cache = False

    return metrics, pd.DataFrame(rows)


# 14. 학습 전 추론 Sanity Check

In [ ]:
model.eval()
model.config.use_cache = True

row = dev_df.iloc[0]

pred, scores, crop_meta = predict_one(
    row,
    model,
    processor,
)

print("Question:", row["question"])
print("Prediction:", pred)
print("Scores:", scores)
print("Crop:", crop_meta)
print("Human labels:", get_valid_dev_labels(row))

assert pred in CHOICES

model.config.use_cache = False


# 15. Training

In [ ]:
BEST_DIR = os.path.join(
    DRIVE_PROJECT_DIR,
    "best_lora",
)

LAST_DIR = os.path.join(
    DRIVE_PROJECT_DIR,
    "last_lora",
)

HISTORY_PATH = os.path.join(
    DRIVE_PROJECT_DIR,
    "training_history.csv",
)

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

total_updates = EPOCHS * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_updates * WARMUP_RATIO),
    num_training_steps=total_updates,
)

optimizer.zero_grad(set_to_none=True)

best_score = -1.0
best_epoch = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    model.config.use_cache = False

    epoch_loss_sum = 0.0
    batch_count = 0

    pbar = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{EPOCHS} [train]",
    )

    for step, batch in pbar:
        batch = move_to_gpu(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            outputs = model(**batch)
            raw_loss = outputs.loss
            loss = raw_loss / GRAD_ACCUM

        loss.backward()

        epoch_loss_sum += float(raw_loss.detach().item())
        batch_count += 1

        if (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM,
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix({
            "loss": f"{raw_loss.item():.4f}",
            "VRAM": f"{torch.cuda.memory_allocated()/(1024**3):.1f}GB",
        })

    train_loss = epoch_loss_sum / max(batch_count, 1)

    metrics, dev_predictions = evaluate_dev(
        model,
        processor,
        dev_df,
        desc=f"Epoch {epoch} [dev]",
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        **metrics,
    }
    history.append(record)

    print()
    print(
        f"Epoch {epoch} | "
        f"loss={train_loss:.6f} | "
        f"consensus={metrics['consensus_score']:.6f} | "
        f"majority={metrics['majority_accuracy']:.6f} | "
        f"high_conf={metrics['high_conf_accuracy']:.6f} | "
        f"crop_usage={metrics['crop_usage_rate']:.3f}"
    )

    dev_predictions.to_csv(
        os.path.join(
            DRIVE_PROJECT_DIR,
            f"dev_predictions_epoch_{epoch}.csv",
        ),
        index=False,
        encoding="utf-8-sig",
    )

    if metrics["consensus_score"] > best_score:
        best_score = metrics["consensus_score"]
        best_epoch = epoch

        if os.path.isdir(BEST_DIR):
            shutil.rmtree(BEST_DIR)

        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)

        print(
            "★ BEST UPDATED:",
            best_epoch,
            best_score,
        )

    if os.path.isdir(LAST_DIR):
        shutil.rmtree(LAST_DIR)

    model.save_pretrained(LAST_DIR)

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False,
        encoding="utf-8-sig",
    )

print()
print("Best epoch:", best_epoch)
print("Best consensus:", best_score)

display(pd.DataFrame(history))


# 16. Best 모델 재로드

In [ ]:
try:
    del optimizer
    del scheduler
    del trainable_params
    del model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda")

base_model.config.use_cache = True

model = PeftModel.from_pretrained(
    base_model,
    BEST_DIR,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

processor = AutoProcessor.from_pretrained(
    BEST_DIR,
    trust_remote_code=True,
)

processor.image_processor.size = {
    "longest_edge": MAX_PIXELS,
    "shortest_edge": MIN_PIXELS,
}

choice_token_ids = {
    ch: processor.tokenizer(
        ch,
        add_special_tokens=False,
    ).input_ids
    for ch in CHOICES
}

print("Best model loaded:", BEST_DIR)


# 17. Test 추론

In [ ]:
test_records = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Test inference",
):
    pred, scores, crop_meta = predict_one(
        row,
        model,
        processor,
    )

    test_records.append({
        "id": row["id"],
        "answer": pred,
        "crop_used": crop_meta.get("used", False),
        "crop_score": crop_meta.get("score", np.nan),
        "score_a": scores["a"],
        "score_b": scores["b"],
        "score_c": scores["c"],
        "score_d": scores["d"],
    })

test_pred_df = pd.DataFrame(test_records)

print(
    "Crop usage rate:",
    test_pred_df["crop_used"].mean(),
)

display(test_pred_df.head())


# 18. submission.csv 생성

In [ ]:
submission = test_pred_df[
    ["id", "answer"]
].copy()

if DEBUG:
    submission_path = os.path.join(
        DRIVE_PROJECT_DIR,
        "DEBUG_submission_DO_NOT_UPLOAD.csv",
    )

    submission.to_csv(
        submission_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("⚠️ DEBUG MODE — Kaggle 제출 금지")
    print("Rows:", len(submission))
    print("Saved:", submission_path)

else:
    assert len(submission) == EXPECTED_FULL_TEST_ROWS
    assert submission["id"].tolist() == test_full_df["id"].tolist()
    assert submission["answer"].isin(CHOICES).all()

    submission_path = os.path.join(
        DRIVE_PROJECT_DIR,
        "submission.csv",
    )

    submission.to_csv(
        submission_path,
        index=False,
        encoding="utf-8-sig",
    )

    test_pred_df.to_csv(
        os.path.join(
            DRIVE_PROJECT_DIR,
            "test_predictions_with_logits_and_crop.csv",
        ),
        index=False,
        encoding="utf-8-sig",
    )

    print("✅ FINAL SUBMISSION READY")
    print("Rows:", len(submission))
    print("Saved:", submission_path)



# 19. V3 결과 기록

현재 최고:

```text
V1.1
1536 / Full Image only
Kaggle Public = 0.95859
```

V3:

```text
1536 / Full Image + Question-aware OCR Crop
```

결과를 비교하세요.

| Version | Visual | Input | Public |
|---|---:|---|---:|
| V1.1 | 1536 | Full | 0.95859 |
| V2-A | 2048 | Full | 0.95770 |
| V2-B | 2048 | Full + OCR Text | 0.95442 |
| **V3** | **1536** | **Full + Related Crop** | **?** |

### V3가 상승하면

다음 후보:

```text
1. Crop ranking 개선
2. Top-2 crop
3. Full-only / Full+Crop probability ensemble
```

### V3가 상승하지 않으면

OCR 계열은 여기서 멈추고:

```text
1. 1536 baseline에서 4-way classification loss
2. V1.1 + V3 prediction ensemble
3. 최종 Train + 신뢰도 높은 Dev 재학습
```

방향으로 넘어가는 것을 권장합니다.



# Troubleshooting

### torchao 0.10.0 오류

```python
%pip uninstall -y torchao
```

후 런타임 재시작.

### EasyOCR import 오류

```python
%pip install -q "easyocr==1.7.2"
```

### Multi-image 처리 오류

V3는 처음으로 Full + Crop 두 이미지를 동시에 사용합니다.

**DEBUG=True에서 DataLoader sanity check와 학습 전 추론까지 성공한 뒤에만 전체 학습을 시작하세요.**

### OOM

```python
BATCH_SIZE = 1
GRAD_ACCUM = 8
```

로 변경합니다. Effective batch size는 8로 유지됩니다.
